In [37]:
"""
Valutazione PERCP sulle common corruptions: CIFAR10-C, CIFAR100-C, ImageNet-C.
Corruzioni: gaussian_noise, defocus_blur, fog, pixelate. Severita': 1-5.
+ "Worst": per ogni punto, tra le 4 corruzioni (a parita' di severita'),
  tiene quella che degrada di piu' p_true.

------------------------------------------------------------------------
DESIGN (stessa logica delle batterie di attacchi precedenti, adattata):
------------------------------------------------------------------------
- Il regressore (qmodel) e' UNO SOLO, allenato UNA VOLTA su un training set
  PULITO preso dallo split di TRAIN di CIFAR10/100/ImageNet (torchvision).
  Non viene mai riallenato per ogni corruzione/severita': allenarlo 20+
  volte (4 corruzioni x 5 severita') sarebbe uno spreco enorme, dato che
  split conformal non richiede un regressore diverso per ogni condizione -
  la garanzia di coverage viene dalla RICALIBRAZIONE di qhat, non dal
  retraining del modello.

- Per calibrazione e test uso gli split UFFICIALI di RobustBench
  (load_cifar10/load_cifar10c, load_cifar100/load_cifar100c,
  load_imagenet/load_imagenetc): sono allineati indice-per-indice (la
  versione "_c" all'indice i e' la corruzione dell'immagine pulita
  all'indice i), quindi non serve corrompere le immagini a mano - e le
  etichette combaciano automaticamente. Metto comunque un assert di
  sanity-check sulle label, cosi' se l'allineamento dovesse rompersi (per
  una versione diversa della libreria) te ne accorgi subito invece di
  avere risultati silenziosamente sbagliati.

- Questi loader di RobustBench pescano dallo split di TEST/VALIDATION,
  che e' per costruzione disgiunto dal train set usato per il regressore
  -> nessun leakage, meglio ancora della situazione con gli attacchi
  (dove calibrazione e training condividevano la stessa pool di partenza,
  anche se poi splittata correttamente).

- LIMITE NOTO: load_imagenetc restituisce al massimo ~5000 esempi per
  corruzione, indipendentemente da quanti ne chiedi (bug/limite noto della
  libreria, vedi RobustBench issue #92). Tienilo a mente scegliendo n_calib/n_test
  per ImageNet (es. 250+250, non 2000+2000).
"""

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
import pandas as pd
import torch.nn.functional as F

from robustbench.data import (
    load_cifar10, load_cifar10c,
    load_cifar100, load_cifar100c,
    load_imagenet, load_imagenetc,
)
from robustbench.utils import load_model as _rb_load_model
from secml.utils import fm
from secml import settings

import os


CORRUPTIONS = ["gaussian_noise", "defocus_blur", "fog", "pixelate"]
SEVERITIES = [1, 2, 3, 4, 5]

_LOADERS = {
    "CIFAR10": dict(clean=load_cifar10, corrupt=load_cifar10c),
    "CIFAR100": dict(clean=load_cifar100, corrupt=load_cifar100c),
    "IMAGENET": dict(clean=load_imagenet, corrupt=load_imagenetc),
}


# ----------------------------------------------------------------------
# 1) Modello robusto (threat_model='corruptions' e' quello pensato apposta
#    per questo benchmark; puoi comunque passare 'Linf' se vuoi vedere come
#    se la cava un modello adversarially-robust sulle corruptions, e' una
#    domanda legittima e spesso fatta in letteratura - "robustezza Linf
#    trasferisce alle corruptions?")
# ----------------------------------------------------------------------
def load_corruption_model(model_name, dataset, threat_model="corruptions", device="cuda"):
    output_dir = fm.join(settings.SECML_MODELS_DIR, "robustbench")
    model = _rb_load_model(
        model_name=model_name, dataset=dataset.lower(),
        threat_model=threat_model, model_dir=output_dir,
    )
    model.eval()
    model.to(device)
    return model

# ----------------------------------------------------------------------
# 2) Training set PULITO per il regressore (split train, mai toccato dai
#    loader di corruzione che invece pescano dal test/val set)
# ----------------------------------------------------------------------
def load_training_set(dataset_name, root, imagenet_train_dir=None):
    dataset_name = dataset_name.upper()
    if dataset_name == "CIFAR10":
        return torchvision.datasets.CIFAR10(root=root, train=True, download=True, transform=T.ToTensor())
    if dataset_name == "CIFAR100":
        return torchvision.datasets.CIFAR100(root=root, train=True, download=True, transform=T.ToTensor())
    if dataset_name == "IMAGENET":

        if imagenet_train_dir is None:
            raise ValueError(
                "Per ImageNet devi specificare imagenet_train_dir"
            )

        transform = T.Compose([
            T.Resize(256),
            T.CenterCrop(224),
            T.ToTensor()
        ])

        return torchvision.datasets.ImageFolder(
            imagenet_train_dir,
            transform=transform
        )
    raise ValueError(f"Dataset {dataset_name} non supportato.")


# ----------------------------------------------------------------------
# 3) Coppie clean/corrotto allineate, per una data corruzione+severita'
# ----------------------------------------------------------------------
def get_clean_corrupt_pair(dataset_name, corruption, severity, n_calib, n_test, data_dir):
    dataset_name = dataset_name.upper()
    loaders = _LOADERS[dataset_name]
    n_total = n_calib + n_test

    X_clean_all, y_all = loaders["clean"](n_examples=n_total, data_dir=data_dir)
    X_corr_all, y_corr_all = loaders["corrupt"](
        n_examples=n_total, corruptions=[corruption], severity=severity,
        data_dir=data_dir, shuffle=False,
    )

    assert torch.equal(y_all[:n_total], y_corr_all[:n_total]), (
        "Le label di clean e corrotto non coincidono: probabile disallineamento "
        "tra load_*() e load_*c(). Verifica shuffle=False su entrambi e la "
        "versione di robustbench installata."
    )

    X_calib_clean, y_calib = X_clean_all[:n_calib], y_all[:n_calib]
    X_test_clean, y_test = X_clean_all[n_calib:n_total], y_all[n_calib:n_total]
    X_calib_corr = X_corr_all[:n_calib]
    X_test_corr = X_corr_all[n_calib:n_total]

    return X_calib_clean, y_calib, X_calib_corr, X_test_clean, y_test, X_test_corr


# ----------------------------------------------------------------------
# 4) Training del regressore - UNA VOLTA SOLA
# ----------------------------------------------------------------------
def train_percp_regressor(model_rb, QuantileRegressor, ProbabilityRegressionDataset,
                           cqr_loss, get_true_probabilities,
                           X_train, y_train, alpha=0.1, epochs=50, device="cuda"):
    p_train = get_true_probabilities(model_rb, X_train, y_train).view(-1)
    train_loader = torch.utils.data.DataLoader(
        ProbabilityRegressionDataset(X_train, p_train.cpu()), batch_size=32, shuffle=True
    )
    qmodel = QuantileRegressor().to(device)
    optimizer = torch.optim.Adam(qmodel.parameters(), lr=1e-4)
    for epoch in range(epochs):
        qmodel.train()
        total_loss = 0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            pred = qmodel(Xb)
            loss = cqr_loss(pred, yb, alpha)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item()
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"[Regressor] Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")
    qmodel.eval()
    return qmodel


# ----------------------------------------------------------------------
# 5) Valutazione PERCP per UNA coppia clean/corrotto gia' pronta
# ----------------------------------------------------------------------
def evaluate_corruption_percp(qmodel, model_rb, get_true_probabilities,
                               X_calib_clean, y_calib, X_calib_corr,
                               X_test_clean, y_test, X_test_corr,
                               alpha=0.1, device="cuda"):
    qmodel.eval()

    p_calib_clean = get_true_probabilities(model_rb, X_calib_clean, y_calib).to(device).view(-1)
    p_calib_corr = get_true_probabilities(model_rb, X_calib_corr, y_calib).to(device).view(-1)
    p_test_corr = get_true_probabilities(model_rb, X_test_corr, y_test).to(device).view(-1)

    with torch.no_grad():
        pred_calib_clean = qmodel(X_calib_clean.to(device))
        pred_calib_corr = qmodel(X_calib_corr.to(device))
        pred_test_corr = qmodel(X_test_corr.to(device))

    # VANILLA: qhat da calibrazione pulita
    scores_clean = torch.maximum(pred_calib_clean[:, 0] - p_calib_clean,
                                  p_calib_clean - pred_calib_clean[:, 1])
    qhat_vanilla = torch.quantile(scores_clean, 1 - alpha)
    lo_v = pred_test_corr[:, 0] - qhat_vanilla
    hi_v = pred_test_corr[:, 1] + qhat_vanilla
    coverage_v = ((p_test_corr >= lo_v) & (p_test_corr <= hi_v)).float().mean()
    size_v = (hi_v - lo_v).mean()

    # PERCP: qhat ricalibrato su calibrazione CORROTTA (stessa corruzione/severita' del test)
    scores_corr = torch.maximum(pred_calib_corr[:, 0] - p_calib_corr,
                                 p_calib_corr - pred_calib_corr[:, 1])
    qhat_percp = torch.quantile(scores_corr, 1 - alpha)
    lo_p = pred_test_corr[:, 0] - qhat_percp
    hi_p = pred_test_corr[:, 1] + qhat_percp
    coverage_p = ((p_test_corr >= lo_p) & (p_test_corr <= hi_p)).float().mean()
    size_p = (hi_p - lo_p).mean()

    return {
        "coverage_vanilla": coverage_v.item(), "size_vanilla": size_v.item(),
        "coverage_percp": coverage_p.item(), "size_percp": size_p.item(),
        "qhat_vanilla": qhat_vanilla.item(), "qhat_percp": qhat_percp.item(),
    }


class RobustBenchProbabilityRegressor(nn.Module):

    def __init__(self, classifier):
        super().__init__()
        self.classifier = classifier


    def forward(self, x, y):

        logits = self.classifier(x)

        probs = F.softmax(logits, dim=1)

        idx = torch.arange(
            len(y),
            device=x.device
        )

        p_true = probs[idx,y]

        return p_true
    
def get_probability_targets(model,dataset,indices):

    X=[]
    y=[]
    p=[]


    with torch.no_grad():

        for i in indices:

            img,label=dataset[i]

            img=img.unsqueeze(0).cuda()
            label=torch.tensor(
                [label],
                device="cuda"
            )


            prob=model(
                img,
                label
            )


            X.append(img.cpu())
            y.append(label.cpu())
            p.append(prob.cpu())


    return (
        torch.cat(X),
        torch.cat(y),
        torch.cat(p)
    )

def get_true_probabilities(
    classifier,
    X,
    y,
    device="cuda"
):

    classifier.eval()

    probs_all=[]

    batch_size=64

    with torch.no_grad():

        for i in range(0,len(X),batch_size):

            x=X[i:i+batch_size].to(device)
            labels=y[i:i+batch_size].to(device)

            logits=classifier(x)

            probs=torch.softmax(
                logits,
                dim=1
            )

            p_true=probs[
                torch.arange(len(labels),device=device),
                labels
            ]

            probs_all.append(
                p_true.cpu()
            )

    return torch.cat(probs_all)

def split_train_calib(X_cal, y_cal, frac_train=0.5, seed=None):
    """Divide il pool 'X_cal' in due parti disgiunte:
    - X_train/y_train: usati per fittare il quantile regressor
    - X_calib/y_calib: usati SOLO per calcolare qhat (mai visti in training)
    """
    if seed is not None:
        torch.manual_seed(seed)
 
    n = len(X_cal)
    perm = torch.randperm(n)
    n_train = int(n * frac_train)
 
    idx_train = perm[:n_train]
    idx_calib = perm[n_train:]
 
    return (
        X_cal[idx_train], y_cal[idx_train],
        X_cal[idx_calib], y_cal[idx_calib],
    )
 

from torch.utils.data import Dataset, DataLoader


class ProbabilityRegressionDataset(Dataset):

    def __init__(self, X, y):
        self.X = X
        self.y = y.float()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return (
            self.X[idx],
            self.y[idx]
        )
    
import torchvision.models as models


class QuantileRegressor(nn.Module):

    def __init__(self):

        super().__init__()

        self.model = models.resnet18(
            weights=None
        )

        self.model.fc = nn.Linear(
            self.model.fc.in_features,
            2
        )


    def forward(self,x):

        out = self.model(x)

        return out
    
def pinball_loss(pred, target, quantile):

    error = target - pred

    return torch.mean(
        torch.maximum(
            quantile*error,
            (quantile-1)*error
        )
    )


def cqr_loss(pred, target, alpha):
    
    q_low=alpha/2
    q_high=1-alpha/2

    loss_low = pinball_loss(
        pred[:,0],
        target,
        q_low
    )

    loss_high = pinball_loss(
        pred[:,1],
        target,
        q_high
    )

    return loss_low + loss_high    

# ----------------------------------------------------------------------
# 6) "Worst" tra le corruzioni, a parita' di severita'
# ----------------------------------------------------------------------
def evaluate_corruption_worst(qmodel, model_rb, get_true_probabilities, pairs,
                               alpha=0.1, device="cuda"):
    """pairs: dict {corruption_name: (X_calib_clean, y_calib, X_calib_corr,
    X_test_clean, y_test, X_test_corr)} per la STESSA severita'. X_calib_clean/
    y_calib/X_test_clean/y_test sono identici in tutte le entry (stessi punti,
    cambia solo la corruzione applicata) - li prendo dalla prima.
    """
    X_calib_clean, y_calib, _, X_test_clean, y_test, _ = next(iter(pairs.values()))

    calib_candidates, calib_p, test_candidates, test_p = [], [], [], []
    for name, (_, _, X_calib_corr, _, _, X_test_corr) in pairs.items():
        calib_candidates.append(X_calib_corr)
        calib_p.append(get_true_probabilities(model_rb, X_calib_corr, y_calib).view(-1).cpu())
        test_candidates.append(X_test_corr)
        test_p.append(get_true_probabilities(model_rb, X_test_corr, y_test).view(-1).cpu())

    def _pick_worst(candidates, p_list):
        P = torch.stack(p_list, dim=0)          # (n_corruptions, N)
        idx = P.argmin(dim=0)                     # (N,) corruzione peggiore per punto
        stack = torch.stack(candidates, dim=0)     # (n_corruptions, N, C, H, W)
        N = stack.shape[1]
        return stack[idx, torch.arange(N)]

    X_calib_worst = _pick_worst(calib_candidates, calib_p)
    X_test_worst = _pick_worst(test_candidates, test_p)

    return evaluate_corruption_percp(
        qmodel, model_rb, get_true_probabilities,
        X_calib_clean, y_calib, X_calib_worst,
        X_test_clean, y_test, X_test_worst,
        alpha=alpha, device=device,
    )


# ----------------------------------------------------------------------
# 7) Orchestratore: tutta la griglia corruzioni x severita' + Worst
# ----------------------------------------------------------------------
def run_corruption_battery(dataset_name, model_rb, qmodel, get_true_probabilities,
                            n_calib=500, n_test=500, data_dir="./data",
                            corruptions=CORRUPTIONS, severities=SEVERITIES,
                            alpha=0.1, device="cuda"):
    rows = []
    for severity in severities:
        print(f"\n=== Severity {severity} ===")
        pairs = {}
        for corruption in corruptions:
            print(f"--- {corruption} (severity {severity}) ---")
            pair = get_clean_corrupt_pair(dataset_name, corruption, severity, n_calib, n_test, data_dir)
            pairs[corruption] = pair
            X_calib_clean, y_calib, X_calib_corr, X_test_clean, y_test, X_test_corr = pair
            res = evaluate_corruption_percp(
                qmodel, model_rb, get_true_probabilities,
                X_calib_clean, y_calib, X_calib_corr,
                X_test_clean, y_test, X_test_corr,
                alpha=alpha, device=device,
            )
            res.update({"dataset": dataset_name, "corruption": corruption, "severity": severity})
            rows.append(res)

        print(f"--- WORST (severity {severity}) ---")
        res_worst = evaluate_corruption_worst(qmodel, model_rb, get_true_probabilities, pairs,
                                               alpha=alpha, device=device)
        res_worst.update({"dataset": dataset_name, "corruption": "WORST", "severity": severity})
        rows.append(res_worst)

    df = pd.DataFrame(rows)
    df["coverage_vanilla"] *= 100
    df["coverage_percp"] *= 100
    return df




In [39]:
DATASET = "IMAGENET"     # "CIFAR10", "CIFAR100", "IMAGENET"

if DATASET == "CIFAR10":
   
    DATA_DIR = "/home/acarlevaro/Sources/albi/data"
    
elif DATASET == "CIFAR100":
    
    DATA_DIR = "/home/acarlevaro/Sources/albi/OLD/Extension/CIFAR100/data"
    
elif DATASET == "IMAGENET":
    
    DATA_DIR =  "/home/acarlevaro/Sources/albi/OLD/Extension/AttackBench/IMAGENET/ImageNet-C" 



model_rb = load_corruption_model(
    model_name="Tian2022Deeper_DeiT-B",         # o un modello dalla leaderboard 'corruptions' per il tuo dataset
    dataset=DATASET,
    threat_model="corruptions",
)

# training set pulito per il regressore (split TRAIN, disgiunto dal test/val
# che le _c corrompono)
#train_dataset = load_training_set(DATASET, root=DATA_DIR)

train_dataset = load_training_set(
    DATASET,
    root=DATA_DIR,
    imagenet_train_dir="/home/acarlevaro/Sources/albi/OLD/Adversarial_CP_V3/InyImageNet/ILSVRC2012_Albi"
)

regressor = RobustBenchProbabilityRegressor(model_rb).cuda()   # gia' nel tuo notebook

idx_train = torch.randperm(len(train_dataset))[:1000]
X_train, y_train, _ = get_probability_targets(regressor, train_dataset, idx_train)

qmodel = train_percp_regressor(
    model_rb=model_rb,
    QuantileRegressor=QuantileRegressor,
    ProbabilityRegressionDataset=ProbabilityRegressionDataset,
    cqr_loss=cqr_loss,
    get_true_probabilities=get_true_probabilities,
    X_train=X_train, y_train=y_train,
    alpha=0.1, epochs=50,
)

df = run_corruption_battery(
    dataset_name=DATASET,
    model_rb=model_rb,
    qmodel=qmodel,
    get_true_probabilities=get_true_probabilities,
    n_calib=250, n_test=250,     # per IMAGENET usa qualcosa come 250/250 (cap ~5000 di load_imagenetc)
    data_dir=DATA_DIR,
    alpha=0.1,
)
df


[Regressor] Epoch 1/50 - Loss: 0.1535
[Regressor] Epoch 10/50 - Loss: 0.0491
[Regressor] Epoch 20/50 - Loss: 0.0354
[Regressor] Epoch 30/50 - Loss: 0.0290
[Regressor] Epoch 40/50 - Loss: 0.0267
[Regressor] Epoch 50/50 - Loss: 0.0230

=== Severity 1 ===
--- gaussian_noise (severity 1) ---


FileNotFoundError: [Errno 2] No such file or directory: '/home/acarlevaro/Sources/albi/OLD/Extension/AttackBench/IMAGENET/ImageNet-C/val'

In [8]:
df

,coverage_vanilla,size_vanilla,coverage_percp,size_percp,qhat_vanilla,qhat_percp,dataset,corruption,severity
0,79.600006,1.644670,87.600005,1.832791,0.633709,0.727769,CIFAR100,gaussian_noise,1
1,84.800005,1.643252,85.800004,1.688994,0.633709,0.656580,CIFAR100,defocus_blur,1
2,86.400002,1.614655,87.600005,1.644976,0.633709,0.648869,CIFAR100,fog,1
3,84.400004,1.643798,86.000001,1.702479,0.633709,0.663049,CIFAR100,pixelate,1
4,78.600001,1.637780,87.800002,1.839277,0.633709,0.734458,CIFAR100,WORST,1
5,73.200005,1.646452,91.200006,1.942370,0.633709,0.781668,CIFAR100,gaussian_noise,2
6,85.400003,1.641690,86.000001,1.666253,0.633709,0.645991,CIFAR100,defocus_blur,2
7,85.600007,1.619943,90.000004,1.725780,0.633709,0.686628,CIFAR100,fog,2
8,83.600003,1.642718,86.400002,1.759503,0.633709,0.692101,CIFAR100,pixelate,2
9,71.000004,1.632167,93.600005,1.978788,0.633709,0.807019,CIFAR100,WORST,2
